In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import zscore

In [ ]:
def standardized_T1(df_model, MAX_HDRS, MAX_CDI):
    df_model.insert(len(df_model.columns), 'Y_Standardized_T1', None)
    for line_index in df_model.index:
        if pd.notna(df_model.loc[line_index, 'Questionnary1_HDRS_T1_TOT21_Score']):
            df_model.loc[line_index, 'Y_Standardized_T1'] = df_model.loc[line_index, 'Questionnary1_HDRS_T1_TOT21_Score'] / MAX_HDRS
        elif pd.notna(df_model.loc[line_index, 'Questionnary2_CDI_T1_Score']):
            df_model.loc[line_index, 'Y_Standardized_T1'] = (df_model.loc[line_index, 'Questionnary2_CDI_T1_Score'] - 40) / (MAX_CDI - 40)
    return df_model

In [ ]:
def standardized_T1_HDRS(df_model, MAX_HDRS):
    df_model.insert(len(df_model.columns), 'Y_Standardized_T1_HDRS', None)
    for line_index in df_model.index:
        if pd.notna(df_model.loc[line_index, 'Questionnary1_HDRS_T1_TOT21_Score']):
            df_model.loc[line_index, 'Y_Standardized_T1_HDRS'] = df_model.loc[line_index, 'Questionnary1_HDRS_T1_TOT21_Score'] / MAX_HDRS
    return df_model

In [4]:
# Y = T1 - CDI
def standardized_T1_CDI(df_model, MAX_CDI):
    df_model.insert(len(df_model.columns), 'Y_Standardized_T1_CDI', None)
    for line_index in df_model.index:
        if pd.notna(df_model.loc[line_index, 'Questionnary2_CDI_T1_Score']):
            df_model.loc[line_index, 'Y_Standardized_T1_CDI'] = (df_model.loc[line_index, 'Questionnary2_CDI_T1_Score'] - 40) / (MAX_CDI - 40)
    return df_model

In [8]:
# Y = Delta_y (HDRS and CDI) - Regression
def standardized_delta_y(df_model, MAX_HDRS, MAX_CDI):
    df_model.insert(len(df_model.columns), 'Y_Standardized_Delta_Y', None)
    for line_index in df_model.index:
        t0 = t1 = np.nan
        if pd.notna(df_model.loc[line_index, 'Questionnary1_HDRS_T0_TOT21_Score']):
            t0 = df_model.loc[line_index, 'Questionnary1_HDRS_T0_TOT21_Score'] / MAX_HDRS
            t1 = df_model.loc[line_index, 'Questionnary1_HDRS_T1_TOT21_Score'] / MAX_HDRS
        elif pd.notna(df_model.loc[line_index, 'Questionnary2_CDI_T0_Score']):
            t0 = (df_model.loc[line_index, 'Questionnary2_CDI_T0_Score'] - 40) / (MAX_CDI - 40)
            t1 = (df_model.loc[line_index, 'Questionnary2_CDI_T1_Score'] - 40) / (MAX_CDI - 40)
        if pd.notna(t1):
            delta_y = abs(t1 - t0)  
        else:
            delta_y = np.nan  
        df_model.loc[line_index, 'Y_Standardized_Delta_Y'] = delta_y
    return df_model

In [9]:
def calculate_threshold(df_model, MAX_HDRS, MAX_CDI, lower_pct, upper_pct):
    deltas = []
    for idx in df_model.index:
        t0 = t1 = np.nan
        if pd.notna(df_model.loc[idx, 'Questionnary1_HDRS_T0_TOT21_Score']):
            t0 = df_model.loc[idx, 'Questionnary1_HDRS_T0_TOT21_Score'] / MAX_HDRS
            t1 = df_model.loc[idx, 'Questionnary1_HDRS_T1_TOT21_Score'] / MAX_HDRS
        elif pd.notna(df_model.loc[idx, 'Questionnary2_CDI_T0_Score']):
            t0 = (df_model.loc[idx, 'Questionnary2_CDI_T0_Score'] - 40) / (MAX_CDI - 40)
            t1 = (df_model.loc[idx, 'Questionnary2_CDI_T1_Score'] - 40) / (MAX_CDI - 40)
        if pd.notna(t1) and pd.notna(t0):
            deltas.append(t1 - t0)
    lower_threshold = np.percentile(deltas, lower_pct)
    upper_threshold = np.percentile(deltas, upper_pct)
    return lower_threshold, upper_threshold

In [10]:
# Y = Delta_y (HDRS and CDI) - Standardized Binary: worse and better
def standardized_binary_evolution(df_model, MAX_HDRS, MAX_CDI, lower_pct, upper_pct):
    df_model.insert(len(df_model.columns), 'Y_Binary_Classe_Delta_Y', None)
    lower_threshold, upper_threshold = calculate_threshold(df_model, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
    for idx in df_model.index:
        t0 = t1 = np.nan
        if pd.notna(df_model.loc[idx, 'Questionnary1_HDRS_T0_TOT21_Score']):
            t0 = df_model.loc[idx, 'Questionnary1_HDRS_T0_TOT21_Score'] / MAX_HDRS
            t1 = df_model.loc[idx, 'Questionnary1_HDRS_T1_TOT21_Score'] / MAX_HDRS
        elif pd.notna(df_model.loc[idx, 'Questionnary2_CDI_T0_Score']):
            t0 = (df_model.loc[idx, 'Questionnary2_CDI_T0_Score'] - 40) / (MAX_CDI - 40)
            t1 = (df_model.loc[idx, 'Questionnary2_CDI_T1_Score'] - 40) / (MAX_CDI - 40)
        if pd.notna(t1):
            delta_y = t1 - t0
            if delta_y <= lower_threshold:
                df_model.loc[idx, 'Y_Binary_Classe_Delta_Y'] = "Better"
            elif delta_y >= upper_threshold:
                df_model.loc[idx, 'Y_Binary_Classe_Delta_Y'] = "Worse"
        else:
            df_model.loc[idx, 'Y_Binary_Classe_Delta_Y'] = np.nan 
    return df_model

In [12]:
# Y = Delta_y (HDRS and CDI) - Standardized Multiclass: worse, stable, better
def standardized_classify_evolution(df_model, MAX_HDRS, MAX_CDI, lower_pct, upper_pct):
    df_model.insert(len(df_model.columns), 'Y_Classe_Evolution_Delta_Y', None)
    lower_threshold, upper_threshold = calculate_threshold(df_model, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
    for idx in df_model.index:
        t0 = t1 = np.nan
        if pd.notna(df_model.loc[idx, 'Questionnary1_HDRS_T0_TOT21_Score']):
            t0 = df_model.loc[idx, 'Questionnary1_HDRS_T0_TOT21_Score'] / MAX_HDRS
            t1 = df_model.loc[idx, 'Questionnary1_HDRS_T1_TOT21_Score'] / MAX_HDRS
        elif pd.notna(df_model.loc[idx, 'Questionnary2_CDI_T0_Score']):
            t0 = (df_model.loc[idx, 'Questionnary2_CDI_T0_Score'] - 40) / (MAX_CDI - 40)
            t1 = (df_model.loc[idx, 'Questionnary2_CDI_T1_Score'] - 40) / (MAX_CDI - 40)
        if pd.notna(t1):
            delta_y = t1 - t0
            if delta_y <= lower_threshold:
                df_model.loc[idx, 'Y_Classe_Evolution_Delta_Y'] = "Better"
            elif delta_y >= upper_threshold:
                df_model.loc[idx, 'Y_Classe_Evolution_Delta_Y'] = "Worse"
            else:
                df_model.loc[idx, 'Y_Classe_Evolution_Delta_Y'] = "Stable"
        else:
            df_model.loc[idx, 'Y_Classe_Evolution_Delta_Y'] = np.nan 
    return df_model

In [13]:
# Y = Delta_HDRS - Regression
def standardized_delta_HDRS(df_model, MAX_HDRS):
    df_model.insert(len(df_model.columns), 'Y_Standardized_Delta_HDRS', None)
    for line_index in df_model.index:
        t0 = t1 = np.nan
        if pd.notna(df_model.loc[line_index, 'Questionnary1_HDRS_T0_TOT21_Score']):
            t0 = df_model.loc[line_index, 'Questionnary1_HDRS_T0_TOT21_Score'] / MAX_HDRS
            t1 = df_model.loc[line_index, 'Questionnary1_HDRS_T1_TOT21_Score'] / MAX_HDRS
        if pd.notna(t1):
            delta_y = abs(t1 - t0)  
        else:
            delta_y = np.nan  
        df_model.loc[line_index, 'Y_Standardized_Delta_HDRS'] = delta_y
    return df_model

In [14]:
# Y = Delta_HDRS - Standardized Binary: worse and better
def standardized_binary_evolution_HDRS(df_model, MAX_HDRS, MAX_CDI, lower_pct, upper_pct):
    df_model.insert(len(df_model.columns), 'Y_Binary_Classe_HDRS', None)
    lower_threshold, upper_threshold = calculate_threshold(df_model, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
    for idx in df_model.index:
        t0 = t1 = np.nan
        if pd.notna(df_model.loc[idx, 'Questionnary1_HDRS_T0_TOT21_Score']):
            t0 = df_model.loc[idx, 'Questionnary1_HDRS_T0_TOT21_Score'] / MAX_HDRS
            t1 = df_model.loc[idx, 'Questionnary1_HDRS_T1_TOT21_Score'] / MAX_HDRS
        if pd.notna(t1):
            delta_y = t1 - t0
            if delta_y <= lower_threshold:
                df_model.loc[idx, 'Y_Binary_Classe_HDRS'] = "Better"
            elif delta_y >= upper_threshold:
                df_model.loc[idx, 'Y_Binary_Classe_HDRS'] = "Worse"
        else:
            df_model.loc[idx, 'Y_Binary_Classe_HDRS'] = np.nan 
    return df_model

In [15]:
# Y = Delta_HDRS - Standardized Multiclass: worse, stable, better
def standardized_classify_evolution_HDRS(df_model, MAX_HDRS, MAX_CDI, lower_pct, upper_pct):
    df_model.insert(len(df_model.columns), 'Y_Classe_Evolution_HDRS', None)
    lower_threshold, upper_threshold = calculate_threshold(df_model, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
    for idx in df_model.index:
        t0 = t1 = np.nan
        if pd.notna(df_model.loc[idx, 'Questionnary1_HDRS_T0_TOT21_Score']):
            t0 = df_model.loc[idx, 'Questionnary1_HDRS_T0_TOT21_Score'] / MAX_HDRS
            t1 = df_model.loc[idx, 'Questionnary1_HDRS_T1_TOT21_Score'] / MAX_HDRS
        if pd.notna(t1):
            delta_y = t1 - t0
            if delta_y <= lower_threshold:
                df_model.loc[idx, 'Y_Classe_Evolution_HDRS'] = "Better"
            elif delta_y >= upper_threshold:
                df_model.loc[idx, 'Y_Classe_Evolution_HDRS'] = "Worse"
            else:
                df_model.loc[idx, 'Y_Classe_Evolution_HDRS'] = "Stable"
        else:
            df_model.loc[idx, 'Y_Classe_Evolution_HDRS'] = np.nan 
    return df_model

In [16]:
# Y = Delta_CDI - Regression
def standardized_delta_CDI(df_model, MAX_CDI):
    df_model.insert(len(df_model.columns), 'Y_Standardized_Delta_CDI', None)
    for line_index in df_model.index:
        t0 = t1 = np.nan
        if pd.notna(df_model.loc[line_index, 'Questionnary2_CDI_T0_Score']):
            t0 = (df_model.loc[line_index, 'Questionnary2_CDI_T0_Score'] - 40) / (MAX_CDI - 40)
            t1 = (df_model.loc[line_index, 'Questionnary2_CDI_T1_Score'] - 40) / (MAX_CDI - 40)
        if pd.notna(t1):
            delta_y = abs(t1 - t0)  
        else:
            delta_y = np.nan  
        df_model.loc[line_index, 'Y_Standardized_Delta_CDI'] = delta_y
    return df_model

In [17]:
# Y = Delta_CDI - Standardized Binary: worse and better
def standardized_binary_evolution_CDI(df_model, MAX_HDRS, MAX_CDI, lower_pct, upper_pct):
    df_model.insert(len(df_model.columns), 'Y_Binary_Classe_CDI', None)
    lower_threshold, upper_threshold = calculate_threshold(df_model, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
    for idx in df_model.index:
        t0 = t1 = np.nan
        if pd.notna(df_model.loc[idx, 'Questionnary2_CDI_T0_Score']):
            t0 = (df_model.loc[idx, 'Questionnary2_CDI_T0_Score'] - 40) / (MAX_CDI - 40)
            t1 = (df_model.loc[idx, 'Questionnary2_CDI_T1_Score'] - 40) / (MAX_CDI - 40)
        if pd.notna(t1):
            delta_y = t1 - t0 
            if delta_y <= lower_threshold:
                df_model.loc[idx, 'Y_Binary_Classe_CDI'] = "Better"
            elif delta_y >= upper_threshold:
                df_model.loc[idx, 'Y_Binary_Classe_CDI'] = "Worse"
        else:
            df_model.loc[idx, 'Y_Binary_Classe_CDI'] = np.nan 
    return df_model

In [18]:
# Y = Delta_CDI - Standardized Multiclass: worse, stable, better
def standardized_classify_evolution_CDI(df_model, MAX_HDRS, MAX_CDI, lower_pct, upper_pct):
    df_model.insert(len(df_model.columns), 'Y_Classe_Evolution_CDI', None)
    lower_threshold, upper_threshold = calculate_threshold(df_model, MAX_HDRS, MAX_CDI, lower_pct, upper_pct)
    for idx in df_model.index:
        t0 = t1 = np.nan
        if pd.notna(df_model.loc[idx, 'Questionnary2_CDI_T0_Score']):
            t0 = (df_model.loc[idx, 'Questionnary2_CDI_T0_Score'] - 40) / (MAX_CDI - 40)
            t1 = (df_model.loc[idx, 'Questionnary2_CDI_T1_Score'] - 40) / (MAX_CDI - 40)
        if pd.notna(t1):
            delta_y = t1 - t0
            if delta_y <= lower_threshold:
                df_model.loc[idx, 'Y_Classe_Evolution_CDI'] = "Better"
            elif delta_y >= upper_threshold:
                df_model.loc[idx, 'Y_Classe_Evolution_CDI'] = "Worse"
            else:
                df_model.loc[idx, 'Y_Classe_Evolution_CDI'] = "Stable"
        else:
            df_model.loc[idx, 'Y_Classe_Evolution_CDI'] = np.nan 
    return df_model